# Week 02 · Day 02 — Function Calling & Tool Use

**Topics:** OpenAI 4-step tool cycle · Anthropic tool_use · Pydantic structured extraction · Parallel tool calls


In [ ]:
%pip install -q openai anthropic pydantic

In [ ]:
import os, json
from openai import OpenAI
import anthropic

oai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
ant = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

## 1 · OpenAI Tool Use — Complete 4-Step Cycle

The model never executes tools. It only returns the tool name and arguments. Your code executes the tool and sends the result back.

In [ ]:
# Define tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_products",
            "description": "Search the product catalog by keyword and category",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search keywords"},
                    "category": {"type": "string", "enum": ["electronics", "books", "clothing", "all"]},
                    "max_price": {"type": "number", "description": "Maximum price in USD"},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_order_status",
            "description": "Get the current status of an order by order ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "The order ID (e.g. ORD-12345)"},
                },
                "required": ["order_id"],
            },
        },
    },
]

# Mock tool implementations
def search_products(query: str, category: str = "all", max_price: float = None) -> list:
    results = [
        {"id": "P001", "name": "Wireless Headphones", "price": 89.99, "category": "electronics"},
        {"id": "P002", "name": "Bluetooth Speaker", "price": 49.99, "category": "electronics"},
    ]
    if max_price:
        results = [r for r in results if r["price"] <= max_price]
    return results

def get_order_status(order_id: str) -> dict:
    statuses = {
        "ORD-123": {"status": "shipped", "eta": "2025-06-01", "carrier": "FedEx"},
        "ORD-456": {"status": "processing", "eta": "2025-06-03"},
    }
    return statuses.get(order_id, {"status": "not_found"})

TOOL_REGISTRY = {
    "search_products": search_products,
    "get_order_status": get_order_status,
}

def execute_tool(name: str, args: dict) -> str:
    if name not in TOOL_REGISTRY:
        return f"ERROR: Unknown tool '{name}'"
    try:
        return json.dumps(TOOL_REGISTRY[name](**args))
    except Exception as e:
        return f"ERROR: {e}"

print("Tools and registry ready")

In [ ]:
def run_tool_loop(user_message: str) -> str:
    """Complete tool use loop — handles multiple rounds if needed."""
    messages = [{"role": "user", "content": user_message}]

    while True:
        response = oai.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
        )
        msg = response.choices[0].message

        # No tool calls — we have the final answer
        if not msg.tool_calls:
            return msg.content

        # MUST append assistant message before tool results
        messages.append(msg)

        # Execute each tool call
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = execute_tool(tc.function.name, args)
            print(f"  Tool: {tc.function.name}({args}) → {result[:80]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

# Test
queries = [
    "Find me wireless electronics under $100.",
    "What's the status of order ORD-123?",
]

for q in queries:
    print(f"\nUser: {q}")
    answer = run_tool_loop(q)
    print(f"Answer: {answer}")

## 2 · Anthropic Tool Use

Key differences from OpenAI:
- `block.input` is already a dict — no `json.loads()` needed
- Tool result goes in a **user** message (not a `tool` role message)
- `max_tokens` is required

In [ ]:
ant_tools = [
    {
        "name": "get_order_status",
        "description": "Get the current status of an order by order ID",
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "The order ID"}
            },
            "required": ["order_id"],
        },
    }
]

messages = [{"role": "user", "content": "What's the status of order ORD-456?"}]

# Step 1: model decides to use tool
response = ant.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=500,
    tools=ant_tools,
    messages=messages,
)

# Step 2: extract tool call
tool_block = next(b for b in response.content if b.type == "tool_use")
print(f"Tool: {tool_block.name}, Args: {tool_block.input}")  # .input is already a dict!

# Step 3: execute
result = get_order_status(**tool_block.input)

# Step 4: return result in a user message
messages.append({"role": "assistant", "content": response.content})
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_block.id,
        "content": json.dumps(result),
    }]
})

final = ant.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=500,
    tools=ant_tools,
    messages=messages,
)
print(final.content[0].text)

## 3 · Pydantic Structured Extraction

`with_structured_output()` uses tool calling internally but returns a validated Pydantic object directly — no JSON parsing needed.

In [ ]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from typing import Optional

class JobPosting(BaseModel):
    title: str = Field(description="Job title")
    company: str = Field(description="Company name")
    location: str = Field(description="Job location or 'Remote'")
    salary_min: Optional[int] = Field(description="Minimum salary in USD")
    salary_max: Optional[int] = Field(description="Maximum salary in USD")
    required_skills: list[str] = Field(description="List of required technical skills")
    experience_years: Optional[int] = Field(description="Years of experience required")
    remote: bool = Field(description="Whether the role is fully remote")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))
extractor = llm.with_structured_output(JobPosting)

job_text = """
We're hiring a Senior Python Engineer at DataFlow Inc! This is a fully remote role 
based anywhere in the US. You'll need 5+ years of Python experience, proficiency in 
FastAPI, PostgreSQL, and AWS. Bonus points for Kubernetes and Terraform. 
Salary range: $140,000 - $180,000 depending on experience.
"""

result = extractor.invoke(job_text)
print(f"Title: {result.title}")
print(f"Company: {result.company}")
print(f"Remote: {result.remote}")
print(f"Salary: ${result.salary_min:,} - ${result.salary_max:,}")
print(f"Skills: {result.required_skills}")
print(f"Experience: {result.experience_years} years")

In [ ]:
# Batch extraction across multiple job postings
postings = [
    "Junior ML Engineer at StartupAI. On-site in NYC. 0-2 years. Skills: Python, PyTorch. $80k-100k.",
    "Staff Backend Engineer at BigCorp. Hybrid (Seattle). 8+ years Python, Go, Kafka required. $200k-250k.",
    "Data Scientist at Analytics Co. Fully remote. 3 years experience. Python, SQL, Tableau. $110k-140k.",
]

results = [extractor.invoke(p) for p in postings]

print(f"{'Title':<30} {'Remote':<8} {'Min Salary':>12}")
print("-" * 55)
for r in results:
    salary = f"${r.salary_min:,}" if r.salary_min else "N/A"
    print(f"{r.title:<30} {str(r.remote):<8} {salary:>12}")

## 4 · Parallel Tool Calls

In [ ]:
import asyncio
from openai import AsyncOpenAI

async_oai = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

async def run_parallel_tools(user_message: str) -> str:
    messages = [{"role": "user", "content": user_message}]
    response = await async_oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
    )
    msg = response.choices[0].message

    if not msg.tool_calls:
        return msg.content

    messages.append(msg)

    # Execute all tool calls in parallel
    async def run_one(tc):
        args = json.loads(tc.function.arguments)
        result = execute_tool(tc.function.name, args)  # would be async in real code
        return tc.id, result

    results = await asyncio.gather(*[run_one(tc) for tc in msg.tool_calls])

    for tool_call_id, result in results:
        messages.append({"role": "tool", "tool_call_id": tool_call_id, "content": result})

    final = await async_oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools,
    )
    return final.choices[0].message.content

# This will trigger both tools in parallel if the model decides to
answer = asyncio.run(run_parallel_tools(
    "Search for electronics under $60 AND check the status of ORD-123."
))
print(answer)

## Summary

- **4-step OpenAI cycle**: define → call → execute → return result. Always append assistant message first.
- **Anthropic**: `block.input` is a dict (no `json.loads`); tool result is a user message.
- **`with_structured_output()`**: wraps tool calling to return a validated Pydantic model — no parsing.
- **Parallel tools**: use `asyncio.gather()` to execute independent tools concurrently.
- Always return a string from your tool — JSON-serialize dicts and lists.

**Next:** [AI Agents](week02-day03-agents.ipynb)